# Data Loading and Preprocessing



In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
!pip install pypdf
import pypdf
import re
import os
import json
import pandas as pd
import torch
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
from openai import AzureOpenAI
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 21.3 MB/s eta 0:00:00


In [ ]:
cases_config = [
    {
        "name": "Test 1: Redundancy (The Brother Test)",
        "file": "test1_redundant_evidence.pdf",
        "query": "What is the age difference between Kyan Brile and his brother Marcus?",
        "k": 2,
        "residual_alpha": 1,
        "reference": "Must identify Kyan is 42 and Marcus is 38.",
        "chunk_size": 150,
        "overlap": 30
    },
    {
        "name": "Test 2: Sequential Chain (The Logic Test)",
        "file": "test2_sequential_chain.pdf",
        "query": "How does Kyan Brile's genetic mutation lead to increased cancer risk?",
        "k": 3,
        "residual_alpha": 1,
        "reference": "Patient Kyan Brile carries the BRCA1 gene mutation, The BRCA1 gene produces protein BRCA1, Protein BRCA1 repairs DNA damage in cells, When DNA repair fails, cells accumulate mutations, Accumulated mutations increase cancer risk by 60%",
        "chunk_size": 150,
        "overlap": 30
    },
    {
        "name": "Test 3: Multi-Aspect Balance (The Topic Test)",
        "file": "test3_multiaspect_balance.pdf",
        "query": "Provide overview of Drug Zilotram: symptoms, diagnosis, and treatment",
        "k": 3,
        "residual_alpha": 1,
        "reference": "Must cover 3 distinct aspects: 1) Symptoms (Headache, Nausea, Dizziness); 2) Diagnosis (Serum levels 50-150 ng/mL, Liver function tests, ECG); 3) Treatment (Dose reduction, Take with food, Antiemetics).",
        "chunk_size": 150,
        "overlap": 30
    }
]

def read_pdf_text(file_path):
    text = ""
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found. Please upload it!")
        return ""
    try:
        reader = pypdf.PdfReader(file_path)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return text

cleaned_documents = []

print("Loading Validation Datasets...")
for case in cases_config:
    raw_text = read_pdf_text(case['file'])

    clean_text = re.sub(r'\s+', ' ', raw_text).strip()

    case['text'] = clean_text
    cleaned_documents.append(clean_text)
    print(f"Loaded {case['file']} ({len(clean_text)} chars)")

df_documents = cleaned_documents
print(f"\nData loaded! 'df_documents' now contains {len(df_documents)} validation files.")

Loading Validation Datasets...
Loaded test1_redundant_evidence.pdf (8752 chars)
Loaded test2_sequential_chain.pdf (20517 chars)
Loaded test3_multiaspect_balance.pdf (23730 chars)

Data loaded! 'df_documents' now contains 3 validation files.


In [ ]:
file_path = "musique_ans_v1.0_dev.jsonl"
limit_samples = 20

print(f"Loading top {limit_samples} samples from {file_path}...")

real_cases = []

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= limit_samples: break

            row = json.loads(line)
            all_chunks = []
            for p in row['paragraphs']:
                chunk_text = f"Title: {p['title']}\nContent: {p['paragraph_text']}"
                all_chunks.append(chunk_text)

            real_cases.append({
                "name": f"MuSiQue_{row['id']}",
                "query": row['question'],
                "reference": row['answer'],
                "gold_chunks": all_chunks,
                "file": None
            })

    print(f"\nSuccessfully loaded {len(real_cases)} cases.")
    print(f"Sample Query: {real_cases[0]['query']}")
    print(f"Search Space: {len(real_cases[0]['gold_chunks'])} chunks")

except FileNotFoundError:
    print(f"Error: Could not find '{file_path}'. Make sure you dragged it into the Files sidebar!")

Loading top 20 samples from musique_ans_v1.0_dev.jsonl...

Successfully loaded 20 cases.
Sample Query: Who is the spouse of the Green performer?
Search Space: 20 chunks


# Chunking


In [ ]:
def chunk_document(document, chunk_size=500, overlap=50, add_context=False, context_percentage=0.1):
    chunks = []
    doc_len = len(document)
    start = 0

    while start < doc_len:
        end = min(start + chunk_size, doc_len)
        current_chunk = document[start:end]
        chunks.append(current_chunk)

        if end == doc_len:
            break

        start += chunk_size - overlap
        start = max(0, start)

    if add_context:
        contextual_chunks = []
        for i, chunk in enumerate(chunks):
            pre_context = ""
            post_context = ""

            if i > 0:
                prev_chunk = chunks[i-1]
                context_len = int(len(prev_chunk) * context_percentage)
                pre_context = prev_chunk[-context_len:] + " "

            if i < len(chunks) - 1:
                next_chunk = chunks[i+1]
                context_len = int(len(next_chunk) * context_percentage)
                post_context = " " + next_chunk[:context_len]

            contextual_chunks.append(pre_context + chunk + post_context)
        return contextual_chunks

    return chunks

print("\n--- Basic Chunking (chunk_size=150, overlap=30) ---")
chunked_docs_basic = []
for i, doc in enumerate(df_documents):
    doc_chunks = chunk_document(doc, chunk_size=150, overlap=30, add_context=False)
    chunked_docs_basic.extend(doc_chunks)
    print(f"Document {i+1} (first 3 chunks):\n")
    for j, chunk in enumerate(doc_chunks[:3]):
        print(f"Chunk {j+1}: {chunk}")
    print("\n")

print("--- Chunking with Context (chunk_size=150, overlap=30, add_context=True) ---")
chunked_docs_context = []
for i, doc in enumerate(df_documents):
    doc_chunks = chunk_document(doc, chunk_size=150, overlap=30, add_context=True, context_percentage=0.2)
    chunked_docs_context.extend(doc_chunks)
    print(f"Document {i+1} (first 3 chunks with context):\n")
    for j, chunk in enumerate(doc_chunks[:3]):
        print(f"Chunk {j+1}: {chunk}")
    print("\n")

all_chunks = chunked_docs_context

print(f"Total basic chunks generated: {len(chunked_docs_basic)}")
print(f"Total contextual chunks generated: {len(chunked_docs_context)}")



--- Basic Chunking (chunk_size=150, overlap=30) ---
Document 1 (first 3 chunks):

Chunk 1: Medical Records - Patient Information Patient Kyan Brile, aged 42 years, was admitted to the cardiology department for routine examination on March 15
Chunk 2: outine examination on March 15th. The comprehensive physical assessment included electrocardiogram, blood pressure monitoring, and laboratory analysis
Chunk 3: oring, and laboratory analysis of lipid panels. All vital signs were recorded as stable and within normal parameters for a male patient of his age gro


Document 2 (first 3 chunks):

Chunk 1: Genetic Analysis Report - BRCA1 Pathway Patient Kyan Brile carries the BRCA1 gene mutation, identified through comprehensive genomic sequencing perfor
Chunk 2: sive genomic sequencing performed during routine genetic screening procedures. The mutation was detected in exon 11 of the BRCA1 gene located on chrom
Chunk 3: he BRCA1 gene located on chromosome 17, and confirmed through independent va

# Embedding


In [ ]:
print("Loading SentenceTransformer model 'all-MiniLM-L6-v2'...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

print(f"Generating embeddings for {len(all_chunks)} chunks...")
chunk_embeddings = model.encode(all_chunks, show_progress_bar=True)
print("Embeddings generated.")

print(f"Shape of generated embeddings: {chunk_embeddings.shape}")

Loading SentenceTransformer model 'all-MiniLM-L6-v2'...
Model loaded successfully.
Generating embeddings for 442 chunks...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings generated.
Shape of generated embeddings: (442, 384)


## Traditional Top-K RAG Retrieval

In [ ]:
def retrieve_top_k_chunks(query, model, chunk_embeddings, all_chunks, k=2):
    query_embedding = model.encode(query, convert_to_tensor=True)

    if not isinstance(chunk_embeddings, torch.Tensor):
        chunk_embeddings_tensor = torch.from_numpy(chunk_embeddings).to(query_embedding.device)
    else:
        chunk_embeddings_tensor = chunk_embeddings.to(query_embedding.device)

    cosine_scores = util.cos_sim(query_embedding, chunk_embeddings_tensor)[0]

    top_k_indices = torch.topk(cosine_scores, k=k, largest=True).indices.cpu().numpy()

    retrieved_chunks = []
    for idx in top_k_indices:
        retrieved_chunks.append((all_chunks[idx], cosine_scores[idx].item()))

    return retrieved_chunks

sample_query = "What is the age difference between Kyan Brile and his brother Marcus?"
k_value = 3

print(f"\n--- Retrieving top {k_value} chunks for query: '{sample_query}' ---")
retrieved_results = retrieve_top_k_chunks(sample_query, model, chunk_embeddings, all_chunks, k=k_value)

for i, (chunk_text, score) in enumerate(retrieved_results):
    print(f"Top {i+1} Chunk (Score: {score:.4f}):\n{chunk_text}\n")


--- Retrieving top 3 chunks for query: 'What is the age difference between Kyan Brile and his brother Marcus?' ---
Top 1 Chunk (Score: 0.8371):
opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according to comprehens

Top 2 Chunk (Score: 0.8171):
rtment confirm that Marcus Bri rtment confirm that Marcus Brile is 38 years old and maintains a close relationship with his brother Kyan. Medical history review revealed no known he ry review revealed no known he

Top 3 Chunk (Score: 0.7094):
 contact form maintained in th  contact form maintained in the patient's permanent medical record lists Marcus Brile, age 38 years, as Kyan's brother and designated next of kin for  nd designated next of kin for 



##Dynamic K-Means Clustering Retrieval



In [ ]:
def _to_numpy(emb):
    """Convert torch tensor or numpy array or list to numpy float32 array."""
    if emb is None:
        return None
    try:
        import torch
        if isinstance(emb, torch.Tensor):
            emb = emb.detach().cpu().numpy()
    except Exception:
        pass
    emb = np.asarray(emb, dtype=np.float32)
    if emb.ndim == 1:
        emb = emb.reshape(1, -1)
    return emb

def normalize_rows(x, eps=1e-8):
    """L2-normalize rows of x."""
    x = x.astype(np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.maximum(norms, eps)
    return x / norms

def row_cosine_similarity(matrix, vector):
    """Cosine similarity between each row in matrix and a single vector (both must be normalized)."""
    return np.dot(matrix, vector.ravel())


def dynamic_cluster_embeddings(chunk_embeddings,
                               base_threshold=0.75,
                               normalize=True,
                               verbose=False):
    """
    Create clusters dynamically by scanning chunk_embeddings once.
    An embedding joins the best existing cluster if its cosine similarity to that
    cluster centroid >= base_threshold, otherwise we create a new cluster.

    Returns:
        cluster_labels: array (n_chunks,) int cluster index per chunk
        cluster_centroids: array (n_clusters, dim) normalized centroids
        cluster_counts: list of counts per cluster
    """
    X = _to_numpy(chunk_embeddings)
    if normalize:
        X = normalize_rows(X)

    n, dim = X.shape
    cluster_centroids = []
    cluster_counts = []
    labels = np.full(n, -1, dtype=int)

    for i in range(n):
        x = X[i]
        if len(cluster_centroids) == 0:
            cluster_centroids.append(x.copy())
            cluster_counts.append(1)
            labels[i] = 0
            continue

        centroids_mat = np.vstack(cluster_centroids)
        sims = row_cosine_similarity(centroids_mat, x)
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])

        if best_sim >= base_threshold:
            c = cluster_centroids[best_idx]
            count = cluster_counts[best_idx]
            new_c = (c * count + x) / (count + 1)
            new_c = new_c / (np.linalg.norm(new_c) + 1e-12)
            cluster_centroids[best_idx] = new_c
            cluster_counts[best_idx] += 1
            labels[i] = best_idx
        else:
            cluster_centroids.append(x.copy())
            cluster_counts.append(1)
            labels[i] = len(cluster_centroids) - 1

        if verbose and (i % 1000 == 0 and i > 0):
            print(f"Processed {i}/{n} embeddings, clusters so far: {len(cluster_centroids)}")

    cluster_centroids = np.vstack(cluster_centroids)
    return labels, cluster_centroids, cluster_counts

def retrieve_from_dynamic_clusters(query,
                                   model,
                                   chunk_embeddings,
                                   all_chunks,
                                   cluster_labels,
                                   cluster_centroids,
                                   k=5,
                                   base_threshold=0.65,
                                   adjustment_scale=0.25,
                                   normalize=True,
                                   min_threshold=0.25,
                                   max_threshold=0.95,
                                   redundancy_cutoff=0.95):
    """
    Retrieve top-k chunks using dynamic clusters.

    - Compute cluster-query similarities and adapt each cluster's internal threshold:
        per_cluster_threshold = clamp(base_threshold + adjustment_scale * cluster_query_sim, min_threshold, max_threshold)

    - Allocate k proportionally to cluster sizes, with fallbacks if some clusters
      would get 0 slots or if available items are fewer than allocated slots.

    - Within each cluster, choose top items by cosine similarity to query.
      Uses simple redundancy removal (skip items whose cosine to any already
      selected item > redundancy_cutoff).

    Args:
        query: string or precomputed embedding (if string, model.encode will be used)
        model: encoder with .encode(...) returning numpy or torch tensor
        chunk_embeddings: numpy array or torch tensor shape (n, dim)
        all_chunks: list of text chunks (length n)
        cluster_labels: array (n,) int
        cluster_centroids: array (m, dim)
        k: total items to return
        base_threshold, adjustment_scale: used to compute per-cluster thresholds
        redundancy_cutoff: if selected item is too close to previously selected, skip it

    Returns:
        list of tuples (chunk_text, similarity_score) sorted by descending similarity to query
    """
    X = _to_numpy(chunk_embeddings)
    if normalize:
        X = normalize_rows(X)
    centroids = _to_numpy(cluster_centroids)
    if normalize:
        centroids = normalize_rows(centroids)

    if isinstance(query, str):
        q_emb = model.encode(query, convert_to_tensor=True)
    else:
        q_emb = query
    q_emb = _to_numpy(q_emb)
    if normalize:
        q_emb = normalize_rows(q_emb)
    q = q_emb.ravel()

    n = X.shape[0]
    num_clusters = centroids.shape[0]
    cluster_sizes = np.bincount(cluster_labels, minlength=num_clusters)

    cluster_query_sims = np.dot(centroids, q)

    per_cluster_thresholds = base_threshold + adjustment_scale * cluster_query_sims
    per_cluster_thresholds = np.clip(per_cluster_thresholds, min_threshold, max_threshold)

    proportional = (cluster_sizes / float(n)) * k
    allocated = np.floor(proportional).astype(int)

    total_alloc = int(allocated.sum())
    if total_alloc < k:
        remaining = k - total_alloc
        order = np.argsort(cluster_query_sims)[::-1]
        for idx in order:
            if remaining <= 0:
                break
            can_take = max(1, min(cluster_sizes[idx] - allocated[idx], remaining))
            if can_take <= 0:
                continue
            allocated[idx] += 1
            remaining -= 1
    elif total_alloc > k:
        surplus = total_alloc - k
        order = np.argsort(cluster_query_sims)
        for idx in order:
            if surplus <= 0:
                break
            reducible = min(allocated[idx], surplus)
            allocated[idx] -= reducible
            surplus -= reducible

    retrieved = []
    selected_embeddings = []

    for cluster_idx in range(num_clusters):
        need = allocated[cluster_idx]
        if need <= 0 or cluster_sizes[cluster_idx] == 0:
            continue

        chunk_idxs = np.where(cluster_labels == cluster_idx)[0]
        emb_in_cluster = X[chunk_idxs]

        sims = emb_in_cluster.dot(q)
        threshold = per_cluster_thresholds[cluster_idx]
        candidate_mask = sims >= threshold
        if not np.any(candidate_mask):
            candidate_mask = np.ones_like(sims, dtype=bool)

        candidate_indices_local = np.where(candidate_mask)[0]
        sort_order = np.argsort(sims[candidate_indices_local])[::-1]
        chosen = 0
        for idx_local in sort_order:
            if chosen >= need:
                break
            global_idx = chunk_idxs[candidate_indices_local[idx_local]]
            sim_score = float(sims[candidate_indices_local[idx_local]])

            emb_global = X[global_idx]
            skip = False
            for s_emb in selected_embeddings:
                if np.dot(emb_global, s_emb) >= redundancy_cutoff:
                    skip = True
                    break
            if skip:
                continue

            retrieved.append((all_chunks[global_idx], sim_score))
            selected_embeddings.append(emb_global)
            chosen += 1

    if len(retrieved) < k:
        global_sims = X.dot(q)
        selected_texts = set([t for t, _ in retrieved])
        order = np.argsort(global_sims)[::-1]
        for idx in order:
            if len(retrieved) >= k:
                break
            txt = all_chunks[idx]
            if txt in selected_texts:
                continue
            retrieved.append((txt, float(global_sims[idx])))
            selected_texts.add(txt)

    retrieved = sorted(retrieved, key=lambda x: x[1], reverse=True)[:k]
    return retrieved

cluster_labels, cluster_centroids, cluster_counts = dynamic_cluster_embeddings(
    chunk_embeddings,
    base_threshold=0.65,
    normalize=True,
    verbose=False
)

retrieved_results_kmeans = retrieve_from_dynamic_clusters(
    "What is the age difference between Kyan Brile and his brother Marcus?",
    model,
    chunk_embeddings,
    all_chunks,
    cluster_labels,
    cluster_centroids,
    k=2,
    base_threshold=0,
    adjustment_scale=0
)

for i, (txt, score) in enumerate(retrieved_results_kmeans):
    print(f"Top {i+1} (score={score:.4f}): {txt}...")

Top 1 (score=0.8371): opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according to comprehens...
Top 2 (score=0.5557): art rate patterns. The initial art rate patterns. The initial intake form completed during registration confirms that Kyan Brile is 42 years of age, which is entirely consistent wit ich is entirely consistent wit...


In [ ]:
print(f"Cluster labels: {cluster_labels}")
print(f"Shape of cluster centroids: {cluster_centroids.shape}")

Cluster labels: [  0   1   2   0   3   4   0   5   0   6   7   0   8   8   9   0  10  11
  12  13   0  14  15   0  16  17  18  19  20   0  21  22  23  24  24  25
  26  27   0  28   0  29  30  31   0  32  33  34  35  36  37  38  39  40
  41  42  43  44  45  46  47  48  48  49  50  51  52  53  54  55  56  57
  58  59  59  59  60  59  61  62  59  59  63  63  59  59  64  64  65  66
  67  67  68  69  59  63  70  59  71  72  73  74  75  76  59  77  78  79
  80  81  82  83  84  85  85  86  87  88  89  90  91  92  93  94  95  96
  97  98  99 100 101 102 103 104 105  88 106 107 107 108 109 110 110 111
 112 113 114 115 116 117  59 118 118 119 120 121 122 123  59 124  59 125
  59  59 126  59  59  63  63  59 127  59  64  65  65 128  67  67 129  59
  63  67  59  71 130  73  74 131 112  67 132  78 133 134 135 136 137 138
 139  85  86 140  88  89 141  90 142  92  94  95  96  97 143  99 100 101
 144 103 104 145  88 146 107 107 147 109 148 112 111 112 149 113 115 115
 150 117  67 118 151 119 152 153 15

# Residual Retrieval

In [ ]:
def residual_retrieval(query, model, chunk_embeddings, all_chunks, k, alpha=1):
    """
    Residual Retrieval with 'Soft Subtraction' (Alpha).
    alpha=1.0 -> Hard orthogonality (can drift to noise)
    alpha=0.5 -> Soft orthogonality (keeps topic relevance)
    """
    if not isinstance(chunk_embeddings, torch.Tensor):
        chunk_embeddings = torch.tensor(chunk_embeddings, dtype=torch.float32)
    chunk_embs = chunk_embeddings.detach().cpu().float()

    if isinstance(query, str):
        query_emb = model.encode(query, convert_to_tensor=True)
    else:
        query_emb = query

    if not isinstance(query_emb, torch.Tensor):
        query_emb = torch.tensor(query_emb, dtype=torch.float32)

    query_vec = query_emb.squeeze().cpu().float()  # (D,)
    residual = query_vec.clone()

    n = chunk_embs.shape[0]
    selected = []
    used = set()

    def cos_sim_to_residual(res):
        v = res.unsqueeze(0) if res.dim() == 1 else res
        return util.cos_sim(v, chunk_embs)[0]  # (N,)

    for _ in range(min(k, n)):
        sims = cos_sim_to_residual(residual)

        sims_masked = sims.clone()
        for i in used:
            sims_masked[i] = -9999.0

        best_idx = int(torch.argmax(sims_masked).item())
        if sims_masked[best_idx].item() < -9000:
            break

        used.add(best_idx)
        selected.append(best_idx)

        v = chunk_embs[best_idx]
        v_norm_sq = torch.dot(v, v)
        if v_norm_sq.item() > 1e-12:
            projection = (torch.dot(residual, v) / v_norm_sq) * v

            residual = residual - (alpha * projection)

    final_sims = util.cos_sim(query_vec.unsqueeze(0), chunk_embs)[0]

    results = [
        (all_chunks[i], float(final_sims[i].item()))
        for i in selected
    ]

    return results

results = residual_retrieval(
    query="What is the age difference between Kyan Brile and his brother Marcus?",
    model=model,
    chunk_embeddings=chunk_embeddings,
    all_chunks=all_chunks,
    k=3
)

for i, (text, score) in enumerate(results):
    print(f"{i+1}. score={score:.4f}\n{text}\n")


1. score=0.8371
opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according to comprehens

2. score=0.5557
art rate patterns. The initial art rate patterns. The initial intake form completed during registration confirms that Kyan Brile is 42 years of age, which is entirely consistent wit ich is entirely consistent wit

3. score=0.3731
ry review revealed no known he ry review revealed no known hereditary conditions affecting either sibling, though both were counseled regarding age-appropriate preventive screening  ropriate preventive screening 



# Refinement Strategies

In [ ]:
try:
    os.environ['AZURE_KEY'] = userdata.get('AZURE_KEY')
    os.environ['AZURE_ENDPOINT'] = userdata.get('AZURE_ENDPOINT')
    os.environ['AZURE_OPENAI_API_VERSION'] = userdata.get('AZURE_OPENAI_API_VERSION')
    os.environ['AZURE_OPENAI_DEPLOYMENT_NAME'] = userdata.get('AZURE_OPENAI_DEPLOYMENT_NAME')

    client = AzureOpenAI(
        api_key=os.environ['AZURE_KEY'],
        api_version=os.environ['AZURE_OPENAI_API_VERSION'],
        azure_endpoint=os.environ['AZURE_ENDPOINT']
    )

    class LLM:
        def __init__(self, client, deployment_name):
            self.client = client
            self.deployment_name = deployment_name

        def generate(self, prompt):
            try:
                response = self.client.chat.completions.create(
                    model=self.deployment_name,
                    messages=[{"role": "user", "content": prompt}]
                )

                content = response.choices[0].message.content
                if content is None:
                    return "Error: Content Filtered by Azure."

                return content

            except Exception as e:
                return f"Error calling LLM: {str(e)}"

    llm_model_placeholder = LLM(client, os.environ['AZURE_OPENAI_DEPLOYMENT_NAME'])
    print("LLM initialized for GPT-5")

except Exception as e:
    print(f"Setup Error: {e}")

LLM initialized for GPT-5


In [ ]:
def one_shot_refinement(query, retrieved_chunks_with_scores, llm_model):
    combined_chunks_text = " ".join([chunk_text for chunk_text, score in retrieved_chunks_with_scores])
    one_shot_prompt = f"Query: {query}\n\nArticle Information:\n{combined_chunks_text}\n\nBased on the article information, please answer the query. ONLY use what is presented to you in the articles to create your response, do not use knowledge from anywhere else."

    return one_shot_prompt

def iterative_refinement(query, retrieved_chunks_with_scores, llm_model):
    iterative_prompts = []

    first = True
    for chunk_text, score in retrieved_chunks_with_scores:
        if first:
          first = False
          current_iter_prompt = f"Query: {query}\n\nArticle Information:\n{chunk_text}\n\nBased on the article information, please answer the query to the best of your ability. You may ONLY use what is presented to you in the articles to create your response, do not use knowledge from anywhere else."
        else:
          current_iter_prompt = f"Query: {query}\n\n New Article Information:\n{chunk_text}\n\nOld Response:{prev}\n\nBased on this specific piece of context, update previous response if the new article information provides any additional, useful information relevant to answering the question. Only add information that was actually presented in the article, don't use any knowledge you have from anywhere else."
        prev = llm_model.generate(current_iter_prompt)
    iterative_prompt = current_iter_prompt

    return iterative_prompt

print("\n--- Testing One-Shot Refinement ---")
sample_query_refinement = "Tell me about NLP techniques."
sample_retrieved_chunks = retrieved_results_kmeans

one_shot_prompt_output = one_shot_refinement(sample_query_refinement, sample_retrieved_chunks, llm_model_placeholder)
print(f"One-Shot Prompt Output (first 500 chars):\n{one_shot_prompt_output}...")

print("\n--- Testing Iterative Refinement ---")
iterative_prompts_output = iterative_refinement(sample_query_refinement, sample_retrieved_chunks, llm_model_placeholder)
print(f"Iterative Prompt Output (first 500 chars):\n{iterative_prompts_output}...")



--- Testing One-Shot Refinement ---
One-Shot Prompt Output (first 500 chars):
Query: Tell me about NLP techniques.

Article Information:
opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according to comprehens art rate patterns. The initial art rate patterns. The initial intake form completed during registration confirms that Kyan Brile is 42 years of age, which is entirely consistent wit ich is entirely consistent wit

Based on the a...

--- Testing Iterative Refinement ---
Iterative Prompt Output (first 500 chars):
Query: Tell me about NLP techniques.

 New Article Information:
art rate patterns. The initial art rate patterns. The initial intake form completed during registration confirms that Kyan Brile is 42 years of age, which is entirely consistent wit ich is entirely consistent wit

Old Response:The provided article excerpt does not contain

# Execute Experiments and Collect Results


In [ ]:
def run_rag_experiment(query, retrieval_method, retrieval_params, refinement_method,
                       model, chunk_embeddings, all_chunks, llm_model):
    """
    Wrapper to route to the correct retrieval function based on method name.
    """
    retrieved_chunks = []

    if retrieval_method == "traditional":
        results = retrieve_top_k_chunks(query, model, chunk_embeddings, all_chunks, k=retrieval_params['k'])
        retrieved_chunks = results

    elif retrieval_method == "kmeans":
        c_labels, c_centroids, cluster_counts = dynamic_cluster_embeddings(
            chunk_embeddings,
            base_threshold=0.65,
            normalize=True,
            verbose=False
        )
        results = retrieve_from_dynamic_clusters(
            query, model, chunk_embeddings, all_chunks,
            c_labels, c_centroids,
            k=retrieval_params['k'],
            base_threshold=0,
            adjustment_scale=0
        )
        retrieved_chunks = results

    elif retrieval_method == "residual":
        results = residual_retrieval(
            query, model, chunk_embeddings, all_chunks,
            k=retrieval_params['k'],
            alpha=retrieval_params.get('residual_alpha', 1)
        )
        retrieved_chunks = results

    if refinement_method == "one_shot":
        prompt = one_shot_refinement(query, retrieved_chunks, llm_model)
        answer = llm_model.generate(prompt)
    else:
        prompt = iterative_refinement(query, retrieved_chunks, llm_model)
        answer = llm_model.generate(prompt)

    return {
        "retrieved_chunks": retrieved_chunks,
        "llm_generated_answer": answer
    }

def print_retrieved_chunks(retrieved_chunks, method_name):
    """Prints the score and full text for all retrieved chunks."""
    print(f"\n   [Full Retrieved Context for {method_name} (k={len(retrieved_chunks)})]:")
    for i, (txt, score) in enumerate(retrieved_chunks):

        print(f"     -> Chunk {i+1} (Score: {score:.4f}):")
        print(f"        {txt.strip()}")
    print("-" * 60)

In [ ]:
def get_judge_score(query, reference, ans_1, ans_2, ans_3, llm_model):
    prompt = f"""
    You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.

    Task: Compare and rank the three [System Answers] below based on how well they satisfy the [Reference Requirement] for the given [User Query].

    [User Query]: {query}
    [Reference Requirement]: {reference}

    [System Answer 1]: {ans_1}
    [System Answer 2]: {ans_2}
    [System Answer 3]: {ans_3}

    ---
    Evaluation Criteria:
    1. Accuracy: Which answer aligns best with the reference facts?
    2. Completeness: Which answer captures the most key steps/details?
    3. Hallucination: Penalize answers that invent false information.

    Output Format:
    Reasoning: <Brief comparative analysis of the three answers>
    RANKING: <Order from Best to Worst using IDs 1, 2, 3 (e.g., 2 > 1 > 3)>
    """

    try:
        response = llm_model.generate(prompt)
        return response
    except:
        return 0.0, "Error generating judge response"

pd.set_option('display.max_colwidth', None)
results = []


chunks = all_chunks
embeddings = chunk_embeddings

for config in cases_config:
    print(f"\n{'='*60}\nTesting: {config['name']}\n{'='*60}")

    print(f"  -> Traditional (k={config['k']})...")
    m_trad = run_rag_experiment(
        config['query'], "traditional", {'k': config['k']}, "one_shot",
        model, embeddings, chunks, llm_model_placeholder
    )
    print_retrieved_chunks(m_trad['retrieved_chunks'], "Traditional")

    print(f"  -> K-Means...")
    m_kmeans = run_rag_experiment(
        config['query'], "kmeans",  {'k': config['k']}, "one_shot",
        model, embeddings, chunks, llm_model_placeholder
    )
    print_retrieved_chunks(m_kmeans['retrieved_chunks'], "K-Means")

    print(f"  -> Residual (alpha={config['residual_alpha']})...")
    m_residual = run_rag_experiment(
        config['query'], "residual",  {'k': config['k']}, "one_shot",
        model, embeddings, chunks, llm_model_placeholder
    )
    print_retrieved_chunks(m_residual['retrieved_chunks'], "Residual")
    output = get_judge_score(config['query'], config['reference'], m_trad['llm_generated_answer'], m_kmeans['llm_generated_answer'], m_residual['llm_generated_answer'], llm_model_placeholder)
    print(output)
    print(f'Traditional Generated: {m_trad['llm_generated_answer']}')
    print(f'K-Means Generated: {m_kmeans['llm_generated_answer']}')
    print(f'Residual Generated: {m_residual['llm_generated_answer']}')




Testing: Test 1: Redundancy (The Brother Test)
  -> Traditional (k=2)...

   [Full Retrieved Context for Traditional (k=2)]:
     -> Chunk 1 (Score: 0.8371):
        opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according to comprehens
     -> Chunk 2 (Score: 0.8171):
        rtment confirm that Marcus Bri rtment confirm that Marcus Brile is 38 years old and maintains a close relationship with his brother Kyan. Medical history review revealed no known he ry review revealed no known he
------------------------------------------------------------
  -> K-Means...

   [Full Retrieved Context for K-Means (k=2)]:
     -> Chunk 1 (Score: 0.8371):
        opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according t

KeyboardInterrupt: 

In [ ]:
def comp_judge(query, reference, ans_1, ans_2,  llm_model):
    prompt = f"""
    You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.

    Task: Compare and rank the three [System Answers] below based on how well they satisfy the [Reference Requirement] for the given [User Query].

    [User Query]: {query}
    [Reference Requirement]: {reference}

    [System Answer 1]: {ans_1}
    [System Answer 2]: {ans_2}

    ---
    Evaluation Criteria:
    1. Accuracy: Which answer aligns best with the reference facts?
    2. Completeness: Which answer captures the most key steps/details?
    3. Hallucination: Penalize answers that invent false information.

    Output Format:
    Reasoning: <Brief comparative analysis of the 2 answers>
    RANKING: <Ordered comparision between the 2 using their IDs>
    """

    try:
        response = llm_model.generate(prompt)
        return response
    except:
        return 0.0, "Error generating judge response"

pd.set_option('display.max_colwidth', None)
results = []


chunks = all_chunks
embeddings = chunk_embeddings

for config in cases_config:
    print(f"\n{'='*60}\nTesting: {config['name']}\n{'='*60}")

    print(f"  -> Traditional (k={config['k']})...")
    m_trad = run_rag_experiment(
        config['query'], "traditional", {'k': config['k']}, "one_shot",
        model, embeddings, chunks, llm_model_placeholder
    )
    m_trad_2 = run_rag_experiment(
        config['query'], "traditional", {'k': config['k']}, "iterative",
        model, embeddings, chunks, llm_model_placeholder
    )
    print_retrieved_chunks(m_trad['retrieved_chunks'], "Traditional")
    output = comp_judge(config['query'], config['reference'], m_trad['llm_generated_answer'], m_trad_2['llm_generated_answer'], llm_model_placeholder)
    print(output)

    print(f"  -> K-Means...")
    m_kmeans = run_rag_experiment(
        config['query'], "kmeans",  {'k': config['k']}, "one_shot",
        model, embeddings, chunks, llm_model_placeholder
    )
    m_kmeans_2 = run_rag_experiment(
        config['query'], "kmeans",  {'k': config['k']}, "iterative",
        model, embeddings, chunks, llm_model_placeholder
    )
    print_retrieved_chunks(m_kmeans['retrieved_chunks'], "K-Means")
    output = comp_judge(config['query'], config['reference'], m_kmeans['llm_generated_answer'], m_kmeans_2['llm_generated_answer'], llm_model_placeholder)
    print(output)

    print(f"  -> Residual (alpha={config['residual_alpha']})...")
    m_residual = run_rag_experiment(
        config['query'], "residual",  {'k': config['k']}, "one_shot",
        model, embeddings, chunks, llm_model_placeholder
    )
    m_residual_2 = run_rag_experiment(
        config['query'], "residual",  {'k': config['k']}, "iterative",
        model, embeddings, chunks, llm_model_placeholder
    )
    print_retrieved_chunks(m_residual['retrieved_chunks'], "Residual")
    output = comp_judge(config['query'], config['reference'], m_residual['llm_generated_answer'], m_residual_2['llm_generated_answer'], llm_model_placeholder)
    print(output)



Testing: Test 1: Redundancy (The Brother Test)
  -> Traditional (k=2)...

   [Full Retrieved Context for Traditional (k=2)]:
     -> Chunk 1 (Score: 0.8371):
        opriate for his demographic pr opriate for his demographic profile. Marcus Brile, identified as the biological brother of patient Kyan Brile, is 38 years old according to comprehens rs old according to comprehens
     -> Chunk 2 (Score: 0.8171):
        rtment confirm that Marcus Bri rtment confirm that Marcus Brile is 38 years old and maintains a close relationship with his brother Kyan. Medical history review revealed no known he ry review revealed no known he
------------------------------------------------------------
Reasoning: Both answers fail to satisfy the reference requirement (Kyan is 42, Marcus is 38, implying a 4-year difference). Each correctly notes Marcus is 38 but neither identifies Kyan’s age or the age difference. Answer 2 adds an unnecessary claim about a “close relationship,” which is not supported by

KeyboardInterrupt: 

# Debug Script: Printing Chunk Simlarity Sorted

In [ ]:
import torch
from sentence_transformers import util

def retrieve_and_print_sorted_scores(query, model, chunk_embeddings, all_chunks, k=2):
    query_embedding = model.encode(query, convert_to_tensor=True)

    if not isinstance(chunk_embeddings, torch.Tensor):
        chunk_embeddings_tensor = torch.from_numpy(chunk_embeddings).to(query_embedding.device)
    else:
        chunk_embeddings_tensor = chunk_embeddings.to(query_embedding.device)

    cosine_scores = util.cos_sim(query_embedding, chunk_embeddings_tensor)[0]

    scored_indices = []
    for i, score in enumerate(cosine_scores):
        scored_indices.append((i, score.item()))

    scored_indices.sort(key=lambda x: x[1], reverse=True)

    print(f"--- Detail: Similarity scores for all {len(all_chunks)} chunks (Sorted High to Low) ---")
    for i, score_val in scored_indices:
        preview_text = all_chunks[i].replace('\n', ' ')
        print(f"Chunk {i:03d} | Score: {score_val:.4f} | Text: {preview_text}...")
    print("-" * 60)
    retrieved_chunks = []
    for i in range(min(k, len(scored_indices))):
        idx, score_val = scored_indices[i]
        retrieved_chunks.append((all_chunks[idx], score_val))

    return retrieved_chunks

sample_query = "How does Kyan Brile's genetic mutation lead to increased cancer risk?"
k_value = 3

print(f"\n--- Processing Query: '{sample_query}' ---")
retrieved_results = retrieve_and_print_sorted_scores(sample_query, model, chunk_embeddings, all_chunks, k=k_value)

print(f"\n--- Final Top {k_value} Selection ---")
for i, (chunk_text, score) in enumerate(retrieved_results):
    print(f"Top {i+1} Chunk (Score: {score:.4f}):\n{chunk_text}\n")


--- Processing Query: 'How does Kyan Brile's genetic mutation lead to increased cancer risk?' ---
--- Detail: Similarity scores for all 442 chunks (Sorted High to Low) ---
Chunk 073 | Score: 0.6610 | Text: Genetic Analysis Report - BRCA1 Pathway Patient Kyan Brile carries the BRCA1 gene mutation, identified through comprehensive genomic sequencing perfor sive genomic sequencing perfor...
Chunk 158 | Score: 0.6407 | Text: nce-based guidelines with regu nce-based guidelines with regular updates as scientific understanding evolves. Patient Kyan Brile carries the BRCA1 gene mutation, identified through   mutation, identified through ...
Chunk 090 | Score: 0.5776 | Text: s significantly increase cance s significantly increase cancer risk by approximately 60%, particularly for breast and ovarian cancers in individuals with BRCA1 mutations as document th BRCA1 mutations as document...
Chunk 183 | Score: 0.5667 | Text: aneously with BRCA1 analysis i aneously with BRCA1 analysis in comprehensi

# Run on MuSiQue Benchmark
- takes a long time so total of 20 queries only

In [ ]:
benchmark_results = []

print(f"\nStarting Benchmark on {len(real_cases)} MuSiQue Questions...")

for idx, case in enumerate(real_cases):
    print(f"\n{'='*60}\nCase {idx+1}: {case['query'][:80]}...")

    chunks = case['gold_chunks']
    if len(chunks) == 0:
        continue

    chunk_embeddings = model.encode(chunks)
    k_val = 4

    print(f"  -> Traditional (k={k_val})...")
    m_trad = run_rag_experiment(
        case['query'], "traditional", {'k': k_val}, "one_shot",
        model, chunk_embeddings, chunks, llm_model_placeholder
    )

    m_trad_iterative = run_rag_experiment(
        case['query'], "traditional", {'k': k_val}, "iterative",
        model, chunk_embeddings, chunks, llm_model_placeholder
    )

    print_retrieved_chunks(m_trad['retrieved_chunks'], "Traditional")

    print(f"  -> K-Means (k={k_val})...")
    m_kmeans = run_rag_experiment(
        case['query'], "kmeans", {'k': k_val}, "one_shot",
        model, chunk_embeddings, chunks, llm_model_placeholder
    )

    m_kmeans_iterative = run_rag_experiment(
        case['query'], "kmeans", {'k': k_val}, "iterative",
        model, chunk_embeddings, chunks, llm_model_placeholder
    )

    print_retrieved_chunks(m_kmeans['retrieved_chunks'], "K-Means")

    print(f"  -> Residual (k={k_val})...")
    m_residual = run_rag_experiment(
        case['query'], "residual", {'k': k_val}, "one_shot",
        model, chunk_embeddings, chunks, llm_model_placeholder
    )

    m_residual_iterative = run_rag_experiment(
        case['query'], "residual", {'k': k_val}, "iterative",
        model, chunk_embeddings, chunks, llm_model_placeholder
    )

    print_retrieved_chunks(m_residual['retrieved_chunks'], "Residual")

    print(f"\n--- COMPARISON 1: Across Methods (One-Shot) ---")
    cross_method_score = get_judge_score(
        case['query'],
        case['reference'],
        m_trad['llm_generated_answer'],
        m_kmeans['llm_generated_answer'],
        m_residual['llm_generated_answer'],
        llm_model_placeholder
    )
    print(cross_method_score)

    ranking_match = re.search(r'RANKING:\s*(\d)\s*>\s*(\d)\s*>\s*(\d)', cross_method_score)
    method_best = int(ranking_match.group(1)) if ranking_match else None

    print(f"\n--- COMPARISON 2a: Traditional (One-Shot vs Iterative) ---")
    trad_comparison = comp_judge(
        case['query'],
        case['reference'],
        m_trad['llm_generated_answer'],
        m_trad_iterative['llm_generated_answer'],
        llm_model_placeholder
    )
    print(trad_comparison)

    trad_winner = "one_shot" if re.search(r'RANKING:\s*1\s*>\s*2', trad_comparison) else "iterative" if re.search(r'RANKING:\s*2\s*>\s*1', trad_comparison) else None

    print(f"\n--- COMPARISON 2b: K-Means (One-Shot vs Iterative) ---")
    kmeans_comparison = comp_judge(
        case['query'],
        case['reference'],
        m_kmeans['llm_generated_answer'],
        m_kmeans_iterative['llm_generated_answer'],
        llm_model_placeholder
    )
    print(kmeans_comparison)

    kmeans_winner = "one_shot" if re.search(r'RANKING:\s*1\s*>\s*2', kmeans_comparison) else "iterative" if re.search(r'RANKING:\s*2\s*>\s*1', kmeans_comparison) else None

    print(f"\n--- COMPARISON 2c: Residual (One-Shot vs Iterative) ---")
    residual_comparison = comp_judge(
        case['query'],
        case['reference'],
        m_residual['llm_generated_answer'],
        m_residual_iterative['llm_generated_answer'],
        llm_model_placeholder
    )
    print(residual_comparison)

    residual_winner = "one_shot" if re.search(r'RANKING:\s*1\s*>\s*2', residual_comparison) else "iterative" if re.search(r'RANKING:\s*2\s*>\s*1', residual_comparison) else None

    print(f"\n--- Generated Answers Summary ---")
    print(f"Traditional (One-Shot): {m_trad['llm_generated_answer']}...")
    print(f"Traditional (Iterative): {m_trad_iterative['llm_generated_answer']}...")
    print(f"K-Means (One-Shot): {m_kmeans['llm_generated_answer']}...")
    print(f"K-Means (Iterative): {m_kmeans_iterative['llm_generated_answer']}...")
    print(f"Residual (One-Shot): {m_residual['llm_generated_answer']}...")
    print(f"Residual (Iterative): {m_residual_iterative['llm_generated_answer']}...")

    benchmark_results.append({
        'case_id': idx + 1,
        'method_best': method_best,
        'trad_winner': trad_winner,
        'kmeans_winner': kmeans_winner,
        'residual_winner': residual_winner
    })

print("\n" + "="*60)
print("BENCHMARK COMPLETE")
print("="*60)

df_results = pd.DataFrame(benchmark_results)

print("\nBest Algorithm (One-Shot):")
method_names = {1: "Traditional", 2: "K-Means", 3: "Residual"}
method_wins = df_results['method_best'].value_counts()
for method_id in [1, 2, 3]:
    wins = method_wins.get(method_id, 0)
    print(f"  {method_names[method_id]}: {wins} wins ({wins/len(benchmark_results)*100:.1f}%)")

print("\nOne-Shot vs Iterative:")
for method_name, col in [('Traditional', 'trad_winner'), ('K-Means', 'kmeans_winner'), ('Residual', 'residual_winner')]:
    winners = df_results[col].value_counts()
    one_shot = winners.get('one_shot', 0)
    iterative = winners.get('iterative', 0)
    print(f"  {method_name}: One-Shot={one_shot}, Iterative={iterative}")


Starting Benchmark on 20 MuSiQue Questions...

Case 1: Who is the spouse of the Green performer?...
  -> Traditional (k=4)...


KeyboardInterrupt: 